## 📑 HAPS Payer/ NonPayer Waterfall
 
#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/AQA5H)**  
  Comprehensive documentation outlining the business requirements for this report.

##### Authentication

In [0]:
%python
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
%python
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col, regexp_replace

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("example").getOrCreate()


from pyspark.sql import DataFrame
from pyspark.sql.functions import trim

In [0]:
%python
import boto3
import os

dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `p-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d') 

PaymentThresholds          = 10.00
NumOfPayThresholds         = 9.00
PayThres12Month            = 0.80
PayThresAfterMod           = 0.80
AdditionalPayThres12Month  = 0.67

# Create a DataFrame from your reference calendar table
calendar_df = spark.sql(f"SELECT * FROM `{env_prefix}-cdh`.reference.ref_calendar")

# Create a temporary view for the DataFrame
calendar_df.createOrReplaceTempView("calendar_df")

EoMreferenceDate = spark.sql(
f'''
SELECT 
     MAX(cal.CalendarDate) CalendarDate
FROM calendar_df AS cal
WHERE cal.IsBusinessDay = 1
AND cal.CalendarDate <= LAST_DAY(DATE_ADD(MONTH, -1, DATE '{run_date}'))
''').collect()[0][0]

referenceDateForHistorical = spark.sql(
f'''
SELECT CAST(
      CASE WHEN (SELECT DayName FROM calendar_df AS cal 
                 WHERE CalendarDate =  MAKE_DATE(year(CAST({run_date!r} AS DATE)),month(CAST({run_date!r} AS DATE)),14) ) = 'Saturday' 
            THEN (SELECT CAST(NextBusinessDay AS DATE) FROM calendar_df AS cal
                  WHERE CalendarDate =  MAKE_DATE(year(CAST({run_date!r} AS DATE)),month(CAST({run_date!r} AS DATE)),16))
            ELSE (SELECT CAST(NextBusinessDay AS DATE) FROM calendar_df AS cal 
                 WHERE CalendarDate =  MAKE_DATE(year(CAST({run_date!r} AS DATE)),month(CAST({run_date!r} AS DATE)),15)) 
      END AS DATE) AS referenceDate
''').collect()[0][0]

PreviousreferenceDateForHistorical = spark.sql(
f'''
SELECT CAST(
      CASE WHEN (SELECT DayName FROM calendar_df AS cal 
                 WHERE CalendarDate =  MAKE_DATE(year(CAST({EoMreferenceDate!r} AS DATE)),month(CAST({EoMreferenceDate!r} AS DATE)),14) ) = 'Saturday' 
            THEN (SELECT CAST(NextBusinessDay AS DATE) FROM calendar_df AS cal
                  WHERE CalendarDate =  MAKE_DATE(year(CAST({EoMreferenceDate!r} AS DATE)),month(CAST({EoMreferenceDate!r} AS DATE)),16))
            ELSE (SELECT CAST(NextBusinessDay AS DATE) FROM calendar_df AS cal 
                 WHERE CalendarDate =  MAKE_DATE(year(CAST({EoMreferenceDate!r} AS DATE)),month(CAST({EoMreferenceDate!r} AS DATE)),15)) 
      END AS DATE) AS referenceDate
''').collect()[0][0]

SoMReferenceDate = spark.sql(
f'''
SELECT 
     MIN(cal.CalendarDate) CalendarDate
FROM calendar_df AS cal
WHERE cal.IsBusinessDay = 1
AND cal.CalendarDate <= LAST_DAY(DATE_ADD(MONTH, -1, DATE '{run_date}'))
AND cal.CalendarDate > LAST_DAY(DATE_ADD(MONTH, -2, DATE '{run_date}'))
''').collect()[0][0]


##### Schema of the table

In [0]:
%python
table_name = "HAPS_PayerNonPayerWaterfall"

schema = (
            "EoMDate					                                              DATE          ,"	
			"BusinessDate						                                      DATE          ,"
			"ExposureKey						                                      STRING        ,"
			"AgreementKey					                                      STRING        ,"
			"PerimeterFlag						                                      STRING        ,"
			"GBVCurrent					                                              DECIMAL(19,2) ,"
			"GBVPrevious					                                          DECIMAL(19,2) ,"
			"PaymentBefore1M					                                      DECIMAL(19,2) ,"
			"PaymentCollection					                                      DECIMAL(19,2) ,"
			"PaymentLiquidation					                                      DECIMAL(19,2) ,"
			"GBVBeforeMod       				                                      DECIMAL(19,2) ,"
			"GBVAfterMod    					                                      DECIMAL(19,2) ,"
			"ModificationHaircut				                                      DECIMAL(19,2) ,"
			"PayerFlagCurrent						                                  STRING        ,"
			"PayerFlagPrevious				                                          STRING        ,"
			"PaymentShock							                                  INT           ,"
			"ModifiedLast6MonthsFlag				                                  INT           ,"
			"PaymentsLast12Months					                                  DECIMAL(19,2) ,"
			"NumberOfPaymentsAboveSpecificAmountLast12Months						  DECIMAL(19,2) ,"
			"InstallmentLast12Months				                                  DECIMAL(19,2) ,"
			"OperationalBucket				                                          INT           ,"
			"DenouncedDate						                                      DATE          ,"
			"P_NonModifiedNoPaymentsAndAmount					                      INT           ,"
			"P_NonModifiedPaymentAmount							                      INT           ,"
			"P_NonModifiedBucket0						                              INT           ,"
			"P_NewGModification						                                  INT           ,"
			"P_NewNonGModification						                              INT           ,"
			"P_ModifiedPaymentAmountOp2							                      INT           ,"
			"P_ModifiedPaymentAmountOp3						                          INT           ,"
			"NP_Denouncement			                                              INT           ,"
			"NP_MissedPayment				                                          INT           ,"
			"NP_InsufficientPayment				                                      INT           ,"
			"GModified				                                                  INT           ,"
			"ObjectiveStatus							                              STRING        ,"
			"InstallmentAfterModification					                          DECIMAL(19,2) ,"
			"PaymentsAfterModification					                              DECIMAL(19,2) ,"
			"PortfolioName						                                      STRING        ,"
			"Category				                                                  STRING        ,"
			"PayerStartingPeriod 						                              DECIMAL(19,2) ,"
			"PayerNoGInflows						                                  DECIMAL(19,2) ,"
			"PayerNoGNoAndAmountOfPaymentsInflows						              DECIMAL(19,2) ,"
			"PayerNoGAmountOfPaymentsInflows					                      DECIMAL(19,2) ,"
			"PayerNoGBucket0Inflows					                                  DECIMAL(19,2) ,"
			"PayerNoGOtherReasonInflows	                                              DECIMAL(19,2) ,"
			"PayerGInflows			                                                  DECIMAL(19,2) ,"
			"PayerGModificationInflows			                                      DECIMAL(19,2) ,"
			"PayerGAmountOfPaymentsInflows			                                  DECIMAL(19,2) ,"
			"PayerOutflows					                                          DECIMAL(19,2) ,"
			"PayerDenouncementOutflows			                                      DECIMAL(19,2) ,"
			"PayerMissedPaymentsOutflows						                      DECIMAL(19,2) ,"
			"PayerMissedPaymentsPaymentShockOutflows						          DECIMAL(19,2) ,"
			"PayerOtherReasonOutflows			                                      DECIMAL(19,2) ,"
			"PayerPayments				                                              DECIMAL(19,2) ,"
			"PayerHaircut					                                          DECIMAL(19,2) ,"
			"PayerLiquidations					                                      DECIMAL(19,2) ,"
			"PayerSales						                                          DECIMAL(19,2) ,"
			"PayerEndingPeriod					                                      DECIMAL(19,2) ,"
			"NonPayerStartingPeriod						                              DECIMAL(19,2) ,"
      		"NonPayerInflows                                                          DECIMAL(19,2) ,"
      		"NonPayerOutflows                                                         DECIMAL(19,2) ,"
      		"NonPayerPayments                                                         DECIMAL(19,2) ,"
      		"NonPayerHaircut                                                          DECIMAL(19,2) ,"
      		"NonPayerLiquidations                                                     DECIMAL(19,2) ,"
      		"NonPayerWriteoffs                                                        DECIMAL(19,2) ,"
      		"NonPayerSales                                                            DECIMAL(19,2) ,"
      		"NonPayerEndingPeriod                                                     DECIMAL(19,2) ,"
      		"PayerPerimeterDecrease                                                   DECIMAL(19,2) ,"
      		"PayerPerimeterIncrease                                                   DECIMAL(19,2) ,"
      		"PayerBalanceIncrease                                                     DECIMAL(19,2) ,"
      		"NonPayerPerimeterDecrease                                                DECIMAL(19,2) ,"
      		"NonPayerPerimeterIncrease                                                DECIMAL(19,2) ,"
      		"NonPayerAccruedInterest                                                  DECIMAL(19,2) ,"
      		"CommonBalanceDiffrence                                                   DECIMAL(19,2) ,"
          	"InsertTimeStamp 														  TIMESTAMP"																		
         )

In [0]:
partition_columns = ["BusinessDate"]
delivery_table_path = ""
 
# ================== Create Delta Table ==================
delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"

delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    run_date
#### type: ignore
)


##### haps_portfolios

In [0]:
haps_portfolios = spark.sql(
f'''
SELECT DISTINCT A.PortfolioName
FROM `{env_prefix}-reportinputfiles`.repoint.haps_repointportfolios  A
INNER JOIN (SELECT BusinessDate, MAX(IngestionTimeStamp) IngestionTimeStamp, PortfolioName 
            FROM `{env_prefix}-reportinputfiles`.repoint.haps_repointportfolios
            GROUP BY  BusinessDate, PortfolioName) B
ON A.BusinessDate = B.BusinessDate AND A.PortfolioName = B.PortfolioName AND A.IngestionTimeStamp = B.IngestionTimeStamp
WHERE A.Report = 'HAPS_PayerNonPayerWaterfall'
AND A.IsActive = 1 
''' 
)
haps_portfolios.createOrReplaceTempView("haps_portfolios")

##### portfolio

In [0]:
portfolio = spark.sql(
f'''
SELECT distinct P.BusinessDate,  p.PortfolioName, cast(P.AssignmentDate as date) AssignmentDate, p.unit_code_profits, p.prefix
FROM `{env_prefix}-cdh`.primary.portfolio P
INNER JOIN haps_portfolios HP ON P.PortfolioName = HP.PortfolioName 
WHERE P.BusinessDate = {run_date!r}
'''
)
portfolio.createOrReplaceTempView("portfolio")  

##### SegmentationDates

In [0]:
%python
SegmentationDates = spark.sql(
f'''
SELECT B.PortfolioName
      ,B.SegmentationDate
      ,MAX(A._update_timestamp) AS _update_timestamp
FROM  `{env_prefix}-ptools`.assignments.offer_collection A
INNER JOIN (
             SELECT PortfolioName, MAX(BusinessDate) AS SegmentationDate
             FROM `{env_prefix}-ptools`.assignments.offer_collection
             WHERE BusinessDate <= CAST('{referenceDateForHistorical}' AS DATE) AND ExposureKeyWithPrefix IS NOT NULL
               AND BusinessDate > CAST('{PreviousreferenceDateForHistorical}' AS DATE)  
             AND PortfolioName is NOT null 
             GROUP BY PortfolioName
           ) B
ON A.PortfolioName  = B.PortfolioName AND A.BusinessDate = B.SegmentationDate  
INNER JOIN portfolio P ON P.PortfolioName = A.PortfolioName
WHERE A.ExposureKeyWithPrefix IS NOT NULL
GROUP BY B.PortfolioName
        ,B.SegmentationDate      
'''
)

SegmentationDates.createOrReplaceTempView("SegmentationDates")

PreviousSegmentationDates = spark.sql(
f'''
SELECT B.PortfolioName
      ,B.SegmentationDate
      ,MAX(A._update_timestamp) AS _update_timestamp
FROM  `{env_prefix}-ptools`.assignments.offer_collection   A 
INNER JOIN (
             SELECT PortfolioName, MAX(BusinessDate) AS SegmentationDate
             FROM `{env_prefix}-ptools`.assignments.offer_collection
             WHERE BusinessDate <= CAST('{PreviousreferenceDateForHistorical}' AS DATE) AND ExposureKeyWithPrefix IS NOT NULL
               AND PortfolioName is NOT null 
             GROUP BY PortfolioName
           ) B
ON A.PortfolioName  = B.PortfolioName AND A.BusinessDate = B.SegmentationDate
INNER JOIN portfolio P ON P.PortfolioName = A.PortfolioName    
WHERE A.ExposureKeyWithPrefix IS NOT NULL
GROUP BY B.PortfolioName
        ,B.SegmentationDate 
'''
)

PreviousSegmentationDates.createOrReplaceTempView("PreviousSegmentationDates")

##### Select Tables

In [0]:
%python
PreviousAssignmentSegmentation = spark.sql(
f'''
SELECT 
	  E.BusinessDate
    , E.ExposureKeyWithPrefix
	, E.Borrower_Payer_Segment        	AS CustomerPayerSegment
    , E.AgreementKey
    , E.ExposureKey
FROM `{env_prefix}-ptools`.assignments.offer_collection E
INNER JOIN PreviousSegmentationDates as SD ON E.BusinessDate = SD.SegmentationDate AND E.PortfolioName = SD.PortfolioName AND E._update_timestamp = SD._update_timestamp
INNER JOIN portfolio P ON P.PortfolioName = E.PortfolioName
WHERE E.RelationType = 'Primary Owner'
'''
)

PreviousAssignmentSegmentation.createOrReplaceTempView("PreviousAssignmentSegmentation")

AssignmentSegmentation = spark.sql(
f'''
SELECT 
	  E.BusinessDate
    , E.ExposureKeyWithPrefix
	, E.Borrower_Payer_Segment        	AS CustomerPayerSegment
    , E.AgreementKey
    , E.ExposureKey
    , CASE WHEN ModifiedLast6MonthsFlag ='true' THEN 1 ELSE 0 END AS ModifiedLast6MonthsFlag 
    , E.PaymentsLast12months
    , PaymentsTrnxFlagging AS NumberOfPaymentsAboveSpecificAmountLast12Months
    , E.InstallmentLast12Months
    , E.OperationalBucket
    , E.PaymentsAfterModification
    , E.InstallmentAfterModification
FROM `{env_prefix}-ptools`.assignments.offer_collection E
INNER JOIN SegmentationDates as SD ON E.BusinessDate = SD.SegmentationDate AND E.PortfolioName = SD.PortfolioName AND E._update_timestamp = SD._update_timestamp
INNER JOIN portfolio P ON P.PortfolioName = E.PortfolioName
WHERE E.RelationType = 'Primary Owner'
'''
)

AssignmentSegmentation.createOrReplaceTempView("AssignmentSegmentation")

In [0]:
dexposure_active = spark.sql(
f'''
SELECT
	    de.ExposureKey
    , de.BusinessDate
    , de.PortfolioName
    , de.GBV
    , de.ObjectiveStatusCode   AS ObjectiveStatus
    , de.IsActiveFlag
    , de.DenouncedDate
FROM `{env_prefix}-cdh`.derivative.dexposure de
INNER JOIN portfolio P ON P.PortfolioName = de.PortfolioName
WHERE de.BusinessDate IN (DATE({SoMReferenceDate!r}), DATE({EoMreferenceDate!r}))
AND de.IsActiveFlag = true
'''
)
dexposure_active.createOrReplaceTempView("dexposure_active")


dexposure_inactive = spark.sql(
f'''
SELECT
      A.ExposureKey
    , {EoMreferenceDate!r} AS BusinessDate
    , A.PortfolioName
    , A.GBV
    , A.ObjectiveStatusCode   AS ObjectiveStatus
    , A.IsActiveFlag
    , A.DenouncedDate
FROM (SELECT e.*
          , rank() OVER (PARTITION BY e.ExposureKey ORDER BY e.BusinessDate DESC) AS Rank
      FROM `{env_prefix}-cdh`.derivative.dexposure e
      WHERE 1 = 1 -- e.IsActiveFlag = false
        AND e.BusinessDate <= {run_date!r}
      ) AS A
LEFT ANTI JOIN dexposure_active e_pf_act ON e_pf_act.ExposureKey = A.ExposureKey AND e_pf_act.BusinessDate = {EoMreferenceDate!r}
INNER JOIN portfolio P ON P.PortfolioName = A.PortfolioName
WHERE A.Rank = 1
'''
)
dexposure_inactive.createOrReplaceTempView("dexposure_inactive")


dexposureClosed = spark.sql(
f'''
SELECT
	  de.ExposureKey
    , MAX(de.BusinessDate) AS BusinessDate
FROM `{env_prefix}-cdh`.derivative.dexposure de
WHERE de.BusinessDate <=  DATE({EoMreferenceDate!r})
AND de.IsActiveFlag  <>'true'
GROUP BY de.ExposureKey
'''
)
dexposureClosed.createOrReplaceTempView("dexposureClosed")

dexposure = spark.sql(
f'''
SELECT *
FROM dexposure_active
 
UNION ALL
 
SELECT *
FROM dexposure_inactive
 
'''
)
dexposure.createOrReplaceTempView("dexposure")


Modifications = spark.sql(
f'''
SELECT DISTINCT
	    de.NewAgreementKey AS Agreementkey
    , de.OldExposureKey 
    , de.NewExposureKey
    , de.RestructuringDate
FROM `{env_prefix}-cdh`.primary.restructuring de
WHERE de.BusinessDate <=  DATE({run_date!r})
'''
)
Modifications.createOrReplaceTempView("Modifications")


ModMapCur = spark.sql(
f'''
SELECT
	    de.AgreementKey
    , de.OldExposureKey
    , de.NewExposureKey
    , de.RestructuringDate
FROM Modifications                            de
INNER JOIN dexposureClosed                     c  ON de.OldExposureKey = c.ExposureKey
--WHERE de.BusinessDate <=  DATE({EoMreferenceDate!r})
'''
)
ModMapCur.createOrReplaceTempView("ModMapCur")

ExchangeRates = spark.sql(
f'''
SELECT 
        E.BusinessDate  
      , E.OriginalCurrency                   
      , E.ExchangeRate                                       
FROM `{env_prefix}-cdh`.reference.exchangerates as E
where E.businessdate = {run_date!r}
'''
)
ExchangeRates.createOrReplaceTempView("ExchangeRates")


Payments = spark.sql(
f'''
SELECT 
 CASE WHEN LEFT(p.ExposureKey,3)= por.Prefix THEN p.ExposureKey
      ELSE concat(por.Prefix,p.ExposureKey) END AS ExposureKey
,p.PaymentType
,SUM( p.PaymentAmount/ COALESCE(e.ExchangeRate,1)) AS PaymentAmount       
FROM `{env_prefix}-cdh`.primary.payment p
INNER JOIN portfolio por
  ON por.unit_code_profits=p.portfolioKey and p.BusinessDate=por.BusinessDate
LEFT JOIN ExchangeRates e
ON p.Currency = e.OriginalCurrency
AND p.BusinessDate = e.BusinessDate
WHERE p.BusinessDate={run_date!r}
  AND p.PaymentActualDate > LAST_DAY(LAST_DAY(CAST({EoMreferenceDate!r} AS DATE)) - INTERVAL 1 MONTH)
  AND p.PaymentActualDate <= {EoMreferenceDate!r} 
  AND p.SourceSystem='profits'
 GROUP BY CASE WHEN LEFT(p.ExposureKey,3)= por.Prefix THEN p.ExposureKey
            ELSE concat(por.Prefix,p.ExposureKey) END
,p.PaymentType
'''
)
Payments.createOrReplaceTempView("Payments")

Payments2 = spark.sql(
f'''
SELECT 
 p.ExposureKey
,SUM(CASE WHEN p.PaymentType ILIKE 'Είσπραξη από πελάτη%' THEN p.PaymentAmount ELSE 0 END) AS PaymentCollection 
,SUM(CASE WHEN p.PaymentType NOT ILIKE 'Είσπραξη από πελάτη%'	THEN p.PaymentAmount ELSE 0 END) AS PaymentLiquidation       
FROM Payments p
 GROUP BY ExposureKey
'''
)
Payments2.createOrReplaceTempView("Payments2")

PaymentsBefore1Month = spark.sql(
f'''
SELECT 
 CASE WHEN LEFT(p.ExposureKey,3)= por.Prefix THEN p.ExposureKey
      ELSE concat(por.Prefix,p.ExposureKey) END AS ExposureKey
,SUM( p.PaymentAmount/ COALESCE(e.ExchangeRate,1)) AS PaymentBefore1M       
FROM `{env_prefix}-cdh`.primary.payment p
INNER JOIN portfolio por
  ON por.unit_code_profits=p.portfolioKey and p.BusinessDate=por.BusinessDate
LEFT JOIN ExchangeRates e
ON p.Currency = e.OriginalCurrency
AND p.BusinessDate = e.BusinessDate
WHERE p.BusinessDate={run_date!r}
  --AND DATEDIFF(MONTH,p.PaymentActualDate, {EoMreferenceDate!r}) = 0
  AND p.PaymentActualDate > LAST_DAY(LAST_DAY(CAST({EoMreferenceDate!r} AS DATE)) - INTERVAL 1 MONTH)
  AND p.PaymentActualDate <= {EoMreferenceDate!r} 
  AND p.SourceSystem='profits'
 GROUP BY CASE WHEN LEFT(p.ExposureKey,3)= por.Prefix THEN p.ExposureKey
            ELSE concat(por.Prefix,p.ExposureKey) END
'''
)
PaymentsBefore1Month.createOrReplaceTempView("PaymentsBefore1Month")

PaymentSchedule = spark.sql(
f'''
SELECT ExposureKey
		,  InstallmentDate
		,  InstallmentType
		,  SUM(InstallmentAmount) AS Installment_Amount
		,  1 CurrentRecord
FROM `{env_prefix}-cdh`.primary.paymentschedule ps 
INNER JOIN portfolio P ON P.PortfolioName = ps.PortfolioName
WHERE InstallmentDate>'2021-12-31' AND  InstallmentDate NOT BETWEEN '2022-06-01' AND '2022-06-30' AND InstallmentType LIKE 'Installment'
  AND ps.BusinessDate = {run_date!r}
GROUP BY ExposureKey
        ,InstallmentDate
        ,CurrentRecord
        ,InstallmentType
'''
)
PaymentSchedule.createOrReplaceTempView("PaymentSchedule")   


fldt_allapplications = spark.sql(
f'''
SELECT * FROM (
SELECT DISTINCT CONCAT(LEFT(app.unifiedLoan_AccountNumberWithPrefix,3),app.profitsInfo_NewAccountNumber) NewExposureKey
               ,product_HairCutAmount
               ,QualcoAppDisbursedDate
, rank()over(partition by CONCAT(LEFT(app.unifiedLoan_AccountNumberWithPrefix,3),app.profitsInfo_NewAccountNumber) order by QualcoAppDisbursedDate DESC) rank
FROM `{env_prefix}-cdh-delivery`.repoint.haps_fldtallapplications AS app 
WHERE APP.ActivatedApplicationFlag = 1
 AND  app.BusinessDate = {run_date!r}
)A WHERE rank=1
'''
)
fldt_allapplications.createOrReplaceTempView("fldt_allapplications")

##### Modification Fields

In [0]:
GBVAtModificationDate = spark.sql(
f'''
SELECT 
      M.NewExposureKey
    , M.RestructuringDate
    , e.GBV AS GBVAtModificationDate
    --,`{env_prefix}-cdh`.derivative.converttoeur(e.Currency, e.GBV, e.BusinessDate) as GBVAtModificationDateEur
    FROM Modifications M
    INNER JOIN `{env_prefix}-cdh`.derivative.dexposure e
    on e.ExposureKey = M.NewExposureKey
    and e.BusinessDate = M.RestructuringDate
'''
)
GBVAtModificationDate.createOrReplaceTempView("GBVAtModificationDate")  

GBV_per_businessdate = spark.sql(
    f'''
SELECT distinct A.ExposureKey
     , CASE WHEN A.Currency='EUR' THEN IFNULL(A.GBV,0) ELSE IFNULL(A.GBV,0)/B.ExchangeRate END GBVEUR
     , A.GBV
     , A.BusinessDate
     , A.Opendate
     , A.LastRestructuringDate
     , A.ClosedDate
FROM `{env_prefix}-cdh`.derivative.dexposure A
LEFT JOIN `{env_prefix}-cdh`.reference.exchangerates B ON A.BusinessDate=B.BusinessDate AND B.OriginalCurrency=A.Currency
'''
)
GBV_per_businessdate.createOrReplaceTempView("GBV_per_businessdate")

GBVAtModificationDate_0 = spark.sql(
    f'''
SELECT *
FROM calendar_df A
INNER JOIN GBV_per_businessdate B ON A.CalendarDate=CASE WHEN B.OpenDate > B.LastRestructuringDate THEN B.OpenDate ELSE B.LastRestructuringDate END
'''
)
GBVAtModificationDate_0.createOrReplaceTempView("GBVAtModificationDate_0")

GBVBeforeClosingDate_0 = spark.sql(
    f'''
SELECT DISTINCT 
      A.PreviousBusinessDay
    , A.CalendarDate
    , B.ClosedDate
    , B.BusinessDate
    , B.ExposureKey
FROM calendar_df A
INNER JOIN GBV_per_businessdate B ON A.CalendarDate=B.BusinessDate and B.ClosedDate IS NOT NULL
'''
)
GBVBeforeClosingDate_0.createOrReplaceTempView("GBVBeforeClosingDate_0")

GBVAtModificationDate = spark.sql(
    f'''
SELECT DISTINCT ExposureKey
     , GBV AS GBVAtModificationDate
     , BusinessDate
FROM GBVAtModificationDate_0 
wHERE BusinessDate=CASE WHEN IsBusinessDay = 0 THEN NextBusinessDay ELSE CalendarDate END 
'''
)
GBVAtModificationDate.createOrReplaceTempView("GBVAtModificationDate")

GBVBeforeClosingDate = spark.sql(
    f'''
SELECT DISTINCT A.ExposureKey
     , A.GBV AS GBVBeforeClosingDate
     , A.BusinessDate
     , A.GBVEUR AS GBVBeforeClosingDateEUR
FROM GBV_per_businessdate A
LEFT JOIN GBVBeforeClosingDate_0 B ON A.exposurekey=B.exposurekey
WHERE A.BusinessDate=B.PreviousBusinessDay
'''
)
GBVBeforeClosingDate.createOrReplaceTempView("GBVBeforeClosingDate")

##### Calculations

In [0]:
PerimeterFlag_Step0 = spark.sql(
    f'''
SELECT DISTINCT NewExposureKey
FROM ModMapCur 
WHERE RestructuringDate BETWEEN {SoMReferenceDate!r} AND {EoMreferenceDate!r} 
'''
)
PerimeterFlag_Step0.createOrReplaceTempView("PerimeterFlag_Step0") 

AccCur = spark.sql(
    f'''
SELECT a.*
      ,b.GBVAtModificationDate
      ,c.GBVBeforeClosingDate
FROM dexposure  a
LEFT JOIN GBVAtModificationDate                                               b ON a.exposurekey = b.exposurekey
LEFT JOIN GBVBeforeClosingDate                                                c ON a.exposurekey = c.exposurekey
WHERE a.BusinessDate = {EoMreferenceDate!r} 
'''
)
AccCur.createOrReplaceTempView("AccCur")  


AccPre = spark.sql(
    f'''
SELECT a.*
FROM dexposure a
WHERE BusinessDate = {SoMReferenceDate!r} 
'''
)
AccPre.createOrReplaceTempView("AccPre")  

PerimeterFlag = spark.sql(
    f'''
SELECT DISTINCT
				AcC.ExposureKeyWithPrefix AS ExposureKey
		,		AcC.AgreementKey
		,		CASE WHEN AcP.ExposureKeyWithPrefix IS NOT NULL											              THEN '1.CommonPerimeter' 
					   WHEN AcP.ExposureKeyWithPrefix IS NULL AND MMC.NewExposureKey IS NOT NULL	                  THEN '2.NewModified'
						 ELSE '3.OtherCases'  END AS PerimeterFlag
FROM AssignmentSegmentation													                    AcC
LEFT JOIN PreviousAssignmentSegmentation						                                AcP ON AcC.ExposureKeyWithPrefix=AcP.ExposureKeyWithPrefix
LEFT JOIN PerimeterFlag_Step0											                        MMC ON AcC.ExposureKeyWithPrefix=MMC.NewExposureKey 
INNER JOIN AccCur                                                                  	             AC ON AcC.ExposureKeyWithPrefix=AC.ExposureKey 
 '''
)
PerimeterFlag.createOrReplaceTempView("PerimeterFlag")     

ModRel = spark.sql(
    f'''
    
  SELECT A.AgreementKey
        ,A.NewExposureKey
        ,A.OldExposureKey
        ,C.GBV GBVCur
        ,C.GBVAtModificationDate
        ,E.GBV GBVPre
        ,d.GBVBeforeClosingDate
		,CASE WHEN F.PaymentType ILIKE 'Είσπραξη από πελάτη%'	                THEN F.PaymentAmount ELSE 0 END PaymentCurCollection
		,CASE WHEN F.PaymentType NOT ILIKE 'Είσπραξη από πελάτη%'	            THEN F.PaymentAmount ELSE 0 END PaymentCurLiquidation
		,CASE WHEN G.PaymentType ILIKE 'Είσπραξη από πελάτη%'	                THEN G.PaymentAmount ELSE 0 END PaymentPreCollection
		,CASE WHEN G.PaymentType NOT ILIKE 'Είσπραξη από πελάτη%'	            THEN G.PaymentAmount ELSE 0 END PaymentPreLiquidation
		,H.CustomerPayerSegment CustomerPayerSegmentCur
		,I.CustomerPayerSegment CustomerPayerSegmentPre
FROM ModMapCur				                A
INNER JOIN PerimeterFlag    	            B ON A.NewExposureKey		  =B.exposurekey AND B.PerimeterFlag='2.NewModified'
LEFT JOIN AccCur			                C ON A.NewExposureKey		  =C.exposurekey AND C.IsActiveFlag ='true'
LEFT JOIN AccCur			                D ON A.OldExposureKey         =D.exposurekey AND D.IsActiveFlag <>'true'
LEFT JOIN AccPre		     	            E ON A.OldExposureKey         =E.exposurekey	
LEFT JOIN Payments			                F ON A.NewExposureKey		  =F.exposurekey
LEFT JOIN Payments			                G ON A.OldExposureKey         =G.exposurekey
LEFT JOIN AssignmentSegmentation			H ON A.NewExposureKey		  =H.ExposureKeyWithPrefix
LEFT JOIN PreviousAssignmentSegmentation	I ON A.OldExposureKey         =I.ExposureKeyWithPrefix
 '''
  )
ModRel.createOrReplaceTempView("ModRel")   

In [0]:
PayerFlagForMissing_Step0 = spark.sql(
    f'''
    
SELECT AgreementKey
	  ,MAX(CASE WHEN CustomerPayerSegmentPre IS NULL THEN 1 ELSE 0 END) HasNull
	  ,MAX(CASE WHEN CustomerPayerSegmentPre ='Non Payer' THEN 1 ELSE 0 END) HasNonPayer
	  ,MAX(CASE WHEN CustomerPayerSegmentPre ='Payer' THEN 1 ELSE 0 END) HasPayer
FROM ModRel
GROUP BY AgreementKey	
   '''
  )
PayerFlagForMissing_Step0.createOrReplaceTempView("PayerFlagForMissing_Step0")   

PayerFlagForMissing = spark.sql(
    f'''
    
SELECT A.AgreementKey		
	  ,CASE WHEN HasNull=1 AND HasPayer=1 THEN 'Payer' 
			WHEN HasNull=0 AND HasPayer=1 THEN 'Payer'
		    ELSE 'Non Payer' END AS CustomerPayerUpdated
FROM PayerFlagForMissing_Step0  a 
GROUP BY a.AgreementKey
		,CASE WHEN HasNull=1 AND HasPayer=1 THEN 'Payer' 
			  WHEN HasNull=0 AND HasPayer=1 THEN 'Payer'
		      ELSE 'Non Payer' END
   '''
  )
PayerFlagForMissing.createOrReplaceTempView("PayerFlagForMissing")         

previoustobeallocated_step0 = spark.sql(
    f'''
    
SELECT OldExposureKey
      ,GBVPre
      ,GBVBeforeClosingDate
      ,PaymentPreCollection
      ,PaymentPreLiquidation
      ,count(*) AS accounts 
FROM ModRel
GROUP BY OldExposureKey
        ,GBVPre
        ,GBVBeforeClosingDate
        ,PaymentPreCollection
        ,PaymentPreLiquidation
    '''
  )
previoustobeallocated_step0.createOrReplaceTempView("previoustobeallocated_step0")      

previoustobeallocated= spark.sql(
    f'''
    
SELECT OldExposureKey
	  ,CASE WHEN GBVPre=0 THEN 0 ELSE GBVPre/accounts end GBVPreAllocated
	  ,CASE WHEN GBVBeforeClosingDate=0 THEN 0 ELSE GBVBeforeClosingDate/accounts END GBVBeforeClosingDateAllocated
	  ,CASE WHEN PaymentPreCollection=0 THEN 0 ELSE PaymentPreCollection/accounts END PaymentPreCollectionAllocated
	  ,CASE WHEN PaymentPreLiquidation=0 THEN 0 ELSE PaymentPreLiquidation/accounts END PaymentPreLiquidationAllocated
FROM previoustobeallocated_step0
'''
  )
previoustobeallocated.createOrReplaceTempView("previoustobeallocated")   

finalallocation_step0 = spark.sql(
    f'''
    
SELECT 
		A.NewExposureKey
	,	A.OldExposureKey
	,	A.AgreementKey
	,	A.GBVCur
	,	B.GBVPreAllocated
	,	A.GBVAtModificationDate
	,	B.GBVBeforeClosingDateAllocated
	,	A.PaymentCurCollection
	,	B.PaymentPreCollectionAllocated
	,	A.PaymentCurLiquidation
	,	B.PaymentPreLiquidationAllocated
	,	A.CustomerPayerSegmentCur
FROM ModRel A 
LEFT JOIN previoustobeallocated B ON A.OldExposureKey=B.OldExposureKey
'''
  )
finalallocation_step0.createOrReplaceTempView("finalallocation_step0")   


finalallocation = spark.sql(
    f'''
    
SELECT 
		NewExposureKey
	,	A.AgreementKey
	,	GBVCur
	,	CAST(SUM(GBVPreAllocated) AS DECIMAL(19,3)) GBVPreAllocated
	,	GBVAtModificationDate
	,	CAST(SUM(GBVBeforeClosingDateAllocated) AS DECIMAL(19,3))	GBVBeforeClosingDateAllocated
	,	CAST(PaymentCurCollection AS DECIMAL(19,2)) PaymentCurCollection
	,	CAST(SUM(PaymentPreCollectionAllocated) AS DECIMAL(19,3))	PaymentPreCollectionAllocated
	,	CAST(PaymentCurLiquidation AS DECIMAL(19,2)) PaymentCurLiquidation
	,	CAST(SUM(PaymentPreLiquidationAllocated) AS DECIMAL(19,3))	PaymentPreLiquidationAllocated
	,	CustomerPayerSegmentCur
	,	CustomerPayerUpdated CustomerPayerSegmentPre
FROM finalallocation_step0      A	
LEFT JOIN PayerFlagForMissing	B ON A.AgreementKey=B.AgreementKey
GROUP BY NewExposureKey
        ,A.AgreementKey
        ,GBVCur
        ,GBVAtModificationDate
        ,PaymentCurCollection
        ,PaymentCurLiquidation
        ,CustomerPayerSegmentCur
        ,CustomerPayerUpdated
'''
  )
finalallocation.createOrReplaceTempView("finalallocation") 

closedaccounts = spark.sql(
    f'''
    
SELECT  a.ExposureKeyWithPrefix AS ExposureKey
      , a.AgreementKey          AS AgreementKey 
FROM PreviousAssignmentSegmentation		A 
INNER JOIN AccPre		               	B ON A.ExposureKeyWithPrefix=B.ExposureKey
LEFT JOIN  PerimeterFlag	            C ON A.ExposureKeyWithPrefix=C.ExposureKey
LEFT JOIN  ModRel			            D ON A.ExposureKeyWithPrefix=D.OldExposureKey
WHERE C.PerimeterFlag IS NULL and d.OldExposureKey IS NULL
'''
  )
closedaccounts.createOrReplaceTempView("closedaccounts") 

PerimeterFlagFinal = spark.sql(
f'''
SELECT *
FROM PerimeterFlag
 
UNION ALL
 
SELECT *
      ,'4.ClosedAccounts' AS PerimeterFlag
FROM ClosedAccounts
 
'''
)
PerimeterFlagFinal.createOrReplaceTempView("PerimeterFlagFinal")

In [0]:
paymentshock_step0 = spark.sql(
f'''
SELECT DISTINCT  ExposureKey ExposureKey
		       , MIN(InstallmentDate) OVER (PARTITION BY  ExposureKey) AS Current_Installment_Date
		       , MIN(Installment_Amount) OVER (PARTITION BY ExposureKey) AS Current_Installment_Amount
		       , LEAD(InstallmentDate,1) OVER (PARTITION BY ExposureKey ORDER BY installmentdate ASC) AS Next_Installment_Date
		       , LEAD(Installment_Amount,1) OVER (PARTITION BY ExposureKey ORDER BY installmentdate ASC) AS Next_Installment_Amount
FROM PaymentSchedule
'''
)
paymentshock_step0.createOrReplaceTempView("paymentshock_step0")

paymentshock_step1 = spark.sql(
f'''
SELECT ExposureKey
	  ,Current_Installment_Date
	  ,Current_Installment_Amount
	  ,Next_Installment_Date
	  ,Next_Installment_Amount
	  ,DENSE_RANK() over (partition by ExposureKey order by Current_Installment_Date asc) as rnk
FROM paymentshock_step0
'''
)
paymentshock_step1.createOrReplaceTempView("paymentshock_step1")

paymentshock_step2 = spark.sql(
f'''
SELECT ExposureKey
	  ,Current_Installment_Date
	  ,Current_Installment_Amount
	  ,Next_Installment_Date
	  ,Next_Installment_Amount
	  ,rnk
	  ,MIN(next_installment_date) OVER (PARTITION BY ExposureKey) AS first_next_date
FROM paymentshock_step1
WHERE Next_Installment_Amount>1.15*Current_Installment_Amount AND Next_Installment_Amount-Current_Installment_Amount>30  
  AND rnk=1
'''
)
paymentshock_step2.createOrReplaceTempView("paymentshock_step2")  

paymentshock = spark.sql(
f'''
SELECT a.ExposureKey
	  ,a.Current_Installment_Date
	  ,a.Current_Installment_Amount
	  ,a.Next_Installment_Date
	  ,a.Next_Installment_Amount
	  ,a.rnk
	  ,a.first_next_date
FROM paymentshock_step2 a
WHERE a.first_next_date=a.Next_Installment_Date
'''
)
paymentshock.createOrReplaceTempView("paymentshock")  

PaymentShockCurrent = spark.sql(
f'''
SELECT * 
FROM paymentShock 
WHERE last_day(Current_Installment_Date) = last_day(DATE({SoMReferenceDate!r}) - interval 1 month)
  AND last_day(Next_Installment_Date) = last_day(DATE({EoMreferenceDate!r}) - interval 1 month)
'''
)
PaymentShockCurrent.createOrReplaceTempView("PaymentShockCurrent") 

In [0]:
table = spark.sql(
f'''
SELECT  A.*
	   ,CASE WHEN A.PerimeterFlag='1.CommonPerimeter'	THEN IFNULL(E.GBV,0)
			 WHEN A.PerimeterFlag='2.NewModified'		THEN IFNULL(E.GBV,0)
			 WHEN A.PerimeterFlag='3.OtherCases'		THEN IFNULL(E.GBV,0)
		     ELSE 0 END AS GBVCurrent
	   ,CASE WHEN A.PerimeterFlag='1.CommonPerimeter'	THEN IFNULL(F.GBV,0)
			 WHEN A.PerimeterFlag='2.NewModified'		THEN IFNULL(D.GBVPreAllocated,0)
			 WHEN A.PerimeterFlag='4.ClosedAccounts'	THEN IFNULL(F.GBV,0)
		     ELSE 0 END AS GBVPrevious
	    ,IFNULL(P.PaymentBefore1M,0) AS  PaymentBefore1M
		,CASE WHEN A.PerimeterFlag='1.CommonPerimeter'	THEN IFNULL(G.PaymentCollection,0)
			  WHEN A.PerimeterFlag='2.NewModified'		THEN IFNULL(D.PaymentPreCollectionAllocated,0) + IFNULL(D.PaymentCurCollection,0)
														ELSE IFNULL(G.PaymentCollection,0)
			 END AS PaymentCollection	
		,CASE WHEN A.PerimeterFlag='1.CommonPerimeter'	THEN IFNULL(G.PaymentLiquidation,0)
			  WHEN A.PerimeterFlag='2.NewModified'		THEN IFNULL(D.PaymentPreLiquidationAllocated,0) +  IFNULL(D.PaymentCurLiquidation,0)
														ELSE IFNULL(G.PaymentLiquidation,0)
			  END AS PaymentLiquidation
		,CASE WHEN A.PerimeterFlag='2.NewModified'		THEN IFNULL(D.GBVBeforeClosingDateAllocated,0)
														ELSE 0
			  END AS GBVBeforeMod
		,CASE WHEN A.PerimeterFlag='2.NewModified'		THEN IFNULL(D.GBVAtModificationDate,0)
														ELSE 0
			  END AS GBVAfterMod
		,CASE WHEN A.PerimeterFlag='2.NewModified'      THEN FA.product_HairCutAmount
              --WHEN A.PerimeterFlag='2.NewModified'		THEN IFNULL(D.GBVBeforeClosingDateAllocated,0)-IFNULL(D.GBVAtModificationDate,0)
														ELSE 0
			  END AS ModificationHaircut
		,CASE WHEN A.PerimeterFlag='1.CommonPerimeter'	THEN B.CustomerPayerSegment
			  WHEN A.PerimeterFlag='2.NewModified'		THEN B.CustomerPayerSegment
			  WHEN A.PerimeterFlag='3.OtherCases'		THEN B.CustomerPayerSegment
		      ELSE NULL END AS PayerFlagCurrent
		,CASE WHEN A.PerimeterFlag='1.CommonPerimeter'	THEN C.CustomerPayerSegment
			  WHEN A.PerimeterFlag='2.NewModified'		THEN D.CustomerPayerSegmentPre
			  WHEN A.PerimeterFlag='4.ClosedAccounts'	THEN C.CustomerPayerSegment
		      ELSE NULL END AS PayerFlagPrevious
		,CASE WHEN H.ExposureKey IS NOT NULL			THEN 1
														ELSE 0
		     END AS PaymentShock
		,B.ModifiedLast6MonthsFlag
		,B.PaymentsLast12Months
		,B.NumberOfPaymentsAboveSpecificAmountLast12Months
		,B.InstallmentLast12Months
		,B.OperationalBucket
		,E.DenouncedDate
		,CASE WHEN	B.NumberOfPaymentsAboveSpecificAmountLast12Months >= {NumOfPayThresholds!r} 
				AND B.PaymentsLast12Months >= ({AdditionalPayThres12Month!r} * B.InstallmentLast12Months)	
				AND B.InstallmentLast12Months>0																		THEN 1 ELSE 0 END AS P_NonModifiedNoPaymentsAndAmount
	    ,CASE WHEN	B.PaymentsLast12Months >= ({PayThres12Month!r} * B.InstallmentLast12Months) 
				AND B.InstallmentLast12Months>0																		THEN 1 ELSE 0 END AS P_NonModifiedPaymentAmount
	    ,CASE WHEN	B.OperationalBucket = 0																			THEN 1 ELSE 0 END AS P_NonModifiedBucket0
	    ,CASE WHEN RIGHT(A.ExposureKey, LEN(A.ExposureKey)-3 ) LIKE '400%'				AND 
				(E.ObjectiveStatus LIKE '074-G%' OR E.ObjectiveStatus LIKE '074-S%')		AND
				B.ModifiedLast6MonthsFlag=1												AND
				A.PerimeterFlag='2.NewModified'																		THEN 1 ELSE 0 END AS P_NewGModification
	    ,CASE WHEN RIGHT(A.ExposureKey, LEN(A.ExposureKey)-3 ) LIKE '400%'				AND 
				A.PerimeterFlag='2.NewModified'																		THEN 1 ELSE 0 END AS P_NewNonGModification
	    ,CASE WHEN B.ModifiedLast6MonthsFlag = 1 
			   AND B.OperationalBucket<= 2	
			   AND P.PaymentBefore1M>0																	THEN 1 ELSE 0 END AS P_ModifiedPaymentAmountOp2
	    ,CASE WHEN B.ModifiedLast6MonthsFlag = 1 
			  AND B.PaymentsAfterModification >= ({PayThres12Month!r} * B.InstallmentAfterModification) 
			  AND B.OperationalBucket >= 3	
			  AND B.InstallmentAfterModification>0													THEN 1 ELSE 0 END AS P_ModifiedPaymentAmountOp3																	
	    ,CASE WHEN LAST_DAY(e.DenouncedDate)='2022-12-31' THEN 1 ELSE 0 END AS NP_Denouncement
	    ,CASE WHEN IFNULL(P.PaymentBefore1M,0)=0		THEN 1 ELSE 0 END AS NP_MissedPayment
	    ,CASE WHEN IFNULL(B.PaymentsAfterModification,0)<IFNULL(B.InstallmentAfterModification,0) THEN 1 ELSE 0 END AS NP_InsufficientPayment
	    ,CASE WHEN RIGHT(A.ExposureKey, LEN(A.ExposureKey)-3 ) LIKE '400%'				AND 
			(E.ObjectiveStatus LIKE '074-G%' OR E.ObjectiveStatus LIKE '074-S%')		AND
			B.ModifiedLast6MonthsFlag=1																			THEN 1 ELSE 0 END AS GModified
	    ,CASE WHEN e.ObjectiveStatus IS NULL THEN '' ELSE e.ObjectiveStatus END  AS ObjectiveStatus 
	    ,B.InstallmentAfterModification
	    ,B.PaymentsAfterModification
	    ,coalesce(e.PortfolioName,f.PortfolioName) Portfolio
FROM PerimeterFlagFinal				                    A
LEFT JOIN AssignmentSegmentation				        B   ON A.ExposureKey=B.ExposureKeyWithPrefix
LEFT JOIN PreviousAssignmentSegmentation				C   ON A.ExposureKey=C.ExposureKeyWithPrefix
LEFT JOIN finalallocation		                        D   ON A.ExposureKey=D.NewExposureKey
LEFT JOIN AccCur				                        E   ON A.ExposureKey=E.ExposureKey
LEFT JOIN AccPre				                        F   ON A.ExposureKey=F.ExposureKey
LEFT JOIN Payments2			                            G   ON A.ExposureKey=G.ExposureKey
LEFT JOIN PaymentShockCurrent	                        H   ON A.ExposureKey=H.ExposureKey
LEFT JOIN PaymentsBefore1Month                          P   ON A.ExposureKey=P.ExposureKey
LEFT JOIN fldt_allapplications                          FA  ON A.ExposureKey=FA.NewExposureKey
  '''
)
table.createOrReplaceTempView("table") 


table2 = spark.sql(
f'''
SELECT 
	   *
	  ,CASE WHEN PayerFlagCurrent	=	PayerFlagPrevious							THEN '00.NoChange'	
			WHEN PayerFlagCurrent	=	'Non Payer'				AND 
				 PayerFlagPrevious	=	'Payer'					AND
				 NP_Denouncement	=	1											THEN '01.P_NP_Denouncement'
			WHEN PayerFlagCurrent	=	'Non Payer'				AND 
				 PayerFlagPrevious  =	'Payer'					AND
				(NP_MissedPayment   =	1						OR
				 NP_InsufficientPayment	=	1)					AND
				 PaymentShock		=	1											THEN '02a.P_NP_MissedPaymentPaymentShock'
			WHEN PayerFlagCurrent	=	'Non Payer'				AND 
				 PayerFlagPrevious  =	'Payer'					AND
				(NP_MissedPayment	=	1						OR
				 NP_InsufficientPayment =	1)										THEN '02b.P_NP_MissedPaymentNoPaymentShock'
			WHEN PayerFlagCurrent	=	'Non Payer'				AND 
				 PayerFlagPrevious  =	'Payer'										THEN '03.P_NP_OtherReason'
		    WHEN PayerFlagCurrent	=	'Payer'					AND 
			     PayerFlagPrevious	=	'Non Payer'				AND
				 GModified			=	0						AND
				 P_NonModifiedNoPaymentsAndAmount	=	1							THEN '04.NP_P_NG_NoAndAmountOfPayments'
			WHEN PayerFlagCurrent	=	'Payer'					AND 
				 PayerFlagPrevious  =	'Non Payer'				AND
				 GModified          =	0						AND
				 P_NonModifiedPaymentAmount	=	1								    THEN '05.NP_P_NG_AmountOfPayments'
			WHEN PayerFlagCurrent	=	'Payer'					AND 
				 PayerFlagPrevious  =	'Non Payer'				AND
				 GModified			=	0						AND
				 P_NonModifiedBucket0 =	1										    THEN '06.NP_P_NG_Bucket0'
		    WHEN PayerFlagCurrent	=	'Payer'					AND 
				 IFNULL(PayerFlagPrevious,'Non Payer')	=	'Non Payer'	AND
				 GModified	        =	0											THEN '07.NP_P_NG_OtherReason'
			WHEN PayerFlagCurrent	=	'Payer'					AND 
				 PayerFlagPrevious	=	'Non Payer'				AND
				 GModified      	=	1						AND
				(P_NewGModification	=	1						OR
				 P_NewNonGModification				=	1)							 THEN '08.NP_P_G_Modification'
			WHEN PayerFlagCurrent   =	'Payer'					AND 
				 PayerFlagPrevious  =	'Non Payer'				AND
				 GModified          =	1						AND
				(P_ModifiedPaymentAmountOp2	=	1				OR
				 P_ModifiedPaymentAmountOp3 =	1)									 THEN '09.NP_P_G_AmountOfPayments'
			WHEN PayerFlagCurrent	IS NULL							                 THEN '10.Closed'						
			ELSE '11.Check' END AS Category
FROM table
    '''
)
table2.createOrReplaceTempView("table2") 

table3 = spark.sql(
f'''
SELECT 		
	    *
	,   CASE WHEN PayerFlagPrevious	='Payer'						THEN GBVPrevious			ELSE 0 END AS PayerStartingPeriod
	,   CASE WHEN Category	LIKE '%NP_P_NG%'						THEN GBVCurrent				ELSE 0 END AS PayerNoGInflows
	,   CASE WHEN Category	LIKE '04.NP_P_NG_NoAndAmountOfPayments'	THEN GBVCurrent				ELSE 0 END AS PayerNoGNoAndAmountOfPaymentsInflows
	,   CASE WHEN Category	LIKE '05.NP_P_NG_AmountOfPayments'		THEN GBVCurrent				ELSE 0 END AS PayerNoGAmountOfPaymentsInflows
	,   CASE WHEN Category	LIKE '06.NP_P_NG_Bucket0'				THEN GBVCurrent				ELSE 0 END AS PayerNoGBucket0Inflows
	,   CASE WHEN Category	LIKE '07.NP_P_NG_OtherReason'			THEN GBVCurrent				ELSE 0 END AS PayerNoGOtherReasonInflows
	,   CASE WHEN Category	LIKE '%NP_P_G%'							THEN GBVCurrent				ELSE 0 END AS PayerGInflows
	,   CASE WHEN Category	LIKE '08.NP_P_G_Modification'			THEN GBVCurrent				ELSE 0 END AS PayerGModificationInflows
	,   CASE WHEN Category	LIKE '09.NP_P_G_AmountOfPayments'		THEN GBVCurrent				ELSE 0 END AS PayerGAmountOfPaymentsInflows
	,   CASE WHEN Category	LIKE '%P_NP_%'							THEN GBVPrevious			ELSE 0 END AS PayerOutflows
	,   CASE WHEN Category	LIKE '01.P_NP_Denouncement'				THEN GBVPrevious			ELSE 0 END AS PayerDenouncementOutflows
	,   CASE WHEN Category	LIKE '02%'								THEN GBVPrevious			ELSE 0 END AS PayerMissedPaymentsOutflows
	,   CASE WHEN Category	LIKE '02a%'								THEN GBVPrevious			ELSE 0 END AS PayerMissedPaymentsPaymentShockOutflows
	,   CASE WHEN Category	LIKE '03.P_NP_OtherReason'				THEN GBVPrevious			ELSE 0 END AS PayerOtherReasonOutflows
	,	CASE WHEN PayerFlagPrevious	='Payer'						THEN PaymentCollection		ELSE 0 END AS PayerPayments				
	,	CASE WHEN PayerFlagPrevious	='Payer'						THEN ModificationHaircut	ELSE 0 END AS PayerHaircut
	,	CASE WHEN PayerFlagPrevious	='Payer'						THEN PaymentLiquidation		ELSE 0 END AS PayerLiquidations				
	,	0																								   AS PayerSales
	,	CASE WHEN PayerFlagCurrent	='Payer'						THEN GBVCurrent				ELSE 0 END AS PayerEndingPeriod
	,	CASE WHEN PayerFlagPrevious	='Non Payer'					THEN GBVPrevious			ELSE 0 END AS NonPayerStartingPeriod
	,	CASE WHEN Category	LIKE '%P_NP_%'							THEN GBVCurrent				ELSE 0 END AS NonPayerInflows
	,	CASE WHEN Category	LIKE '%NP_P_NG%' 
               OR Category	LIKE '%NP_P_G%'							THEN GBVPrevious			ELSE 0 END AS NonPayerOutflows
	,	CASE WHEN PayerFlagPrevious	='Non Payer'					THEN PaymentCollection		ELSE 0 END AS NonPayerPayments			
	,	CASE WHEN PayerFlagPrevious	='Non Payer'					THEN ModificationHaircut	ELSE 0 END AS NonPayerHaircut
	,	CASE WHEN PayerFlagPrevious	='Non Payer'					THEN PaymentLiquidation		ELSE 0 END AS NonPayerLiquidations			
	,	0																								   AS NonPayerWriteoffs
	,	0																								   AS NonPayerSales
	,	CASE WHEN PayerFlagCurrent ='Non Payer'						THEN GBVCurrent				ELSE 0 END AS NonPayerEndingPeriod
FROM table2
'''
)
table3.createOrReplaceTempView("table3") 

final_0 = spark.sql(
f'''
SELECT LAST_DAY({run_date!r})                                                                                   AS EoMDate
	  ,*
	  ,PayerOutflows+PayerPayments+PayerHaircut+PayerLiquidations+PayerSales								    AS PayerPerimeterDecrease
	  ,PayerEndingPeriod-PayerStartingPeriod+
	   (PayerOutflows+PayerPayments+PayerHaircut+PayerLiquidations+PayerSales)						    	    AS PayerPerimeterIncrease
	  ,PayerEndingPeriod-PayerStartingPeriod+
	   (PayerOutflows+PayerPayments+PayerHaircut+PayerLiquidations+PayerSales)-
	   (PayerNoGInflows+PayerGInflows)																		    AS PayerBalanceIncrease
	  ,NonPayerOutflows+NonPayerPayments+NonPayerHaircut+NonPayerLiquidations+NonPayerWriteoffs+NonPayerSales	AS NonPayerPerimeterDecrease
	  ,NonPayerEndingPeriod-NonPayerStartingPeriod+
	   (NonPayerOutflows+NonPayerPayments+NonPayerHaircut+NonPayerLiquidations+NonPayerWriteoffs+NonPayerSales)	AS NonPayerPerimeterIncrease
	  ,NonPayerEndingPeriod-NonPayerStartingPeriod+
	   (NonPayerOutflows+NonPayerPayments+NonPayerHaircut+NonPayerLiquidations+NonPayerWriteoffs+NonPayerSales)-
	   NonPayerInflows																							AS NonPayerAccruedInterest
	  ,{run_date!r}                                                                                             AS BusinessDate
      , CASE WHEN PerimeterFlag ='1.CommonPerimeter'  THEN GBVPrevious - GBVCurrent
			 ELSE 0 END AS CommonBalanceDiffrence
FROM table3
'''
)
final_0.createOrReplaceTempView("final_0") 

##### final_df

In [0]:
%python
final_df = spark.sql(
f'''
		SELECT CAST(EoMDate					                                     AS DATE           ) AS EoMDate
			,  CAST(BusinessDate						                         AS DATE           ) AS BusinessDate						                        
			,  CAST(ExposureKey						                         	 AS STRING         ) AS ExposureKey						                    
			,  CAST(AgreementKey					                             AS STRING         ) AS AgreementKey					                        
			,  CAST(PerimeterFlag						                         AS STRING         ) AS PerimeterFlag						                    
			,  CAST(GBVCurrent					                                 AS DECIMAL(19,2)  ) AS GBVCurrent					                            
			,  CAST(GBVPrevious					                                 AS DECIMAL(19,2)  ) AS GBVPrevious					                            
			,  CAST(PaymentBefore1M					                             AS DECIMAL(19,2)  ) AS PaymentBefore1M					                        
			,  CAST(PaymentCollection					                         AS DECIMAL(19,2)  ) AS PaymentCollection					                    
			,  CAST(PaymentLiquidation					                         AS DECIMAL(19,2)  ) AS PaymentLiquidation					                    
			,  CAST(GBVBeforeMod       				                             AS DECIMAL(19,2)  ) AS GBVBeforeMod       				                        
			,  CAST(GBVAfterMod    					                             AS DECIMAL(19,2)  ) AS GBVAfterMod    					                        
			,  CAST(ModificationHaircut				                             AS DECIMAL(19,2)  ) AS ModificationHaircut				                        
			,  CAST(PayerFlagCurrent						                     AS STRING         ) AS PayerFlagCurrent						                
			,  CAST(PayerFlagPrevious				                             AS STRING         ) AS PayerFlagPrevious				                        
			,  CAST(PaymentShock							                     AS INT            ) AS PaymentShock							                
			,  CAST(ModifiedLast6MonthsFlag				                         AS INT            ) AS ModifiedLast6MonthsFlag				                
			,  CAST(PaymentsLast12Months					                     AS DECIMAL(19,2)  ) AS PaymentsLast12Months					                
			,  CAST(NumberOfPaymentsAboveSpecificAmountLast12Months				 AS	DECIMAL(19,2)  ) AS NumberOfPaymentsAboveSpecificAmountLast12Months			
			,  CAST(InstallmentLast12Months				                         AS DECIMAL(19,2)  ) AS InstallmentLast12Months				                    
			,  CAST(OperationalBucket				                             AS INT            ) AS OperationalBucket				                        
			,  CAST(DenouncedDate						                         AS DATE           ) AS DenouncedDate						                    
			,  CAST(P_NonModifiedNoPaymentsAndAmount					         AS INT            ) AS P_NonModifiedNoPaymentsAndAmount					    
			,  CAST(P_NonModifiedPaymentAmount							         AS INT            ) AS P_NonModifiedPaymentAmount							    
			,  CAST(P_NonModifiedBucket0						                 AS INT            ) AS P_NonModifiedBucket0						            
			,  CAST(P_NewGModification						                     AS INT            ) AS P_NewGModification						                
			,  CAST(P_NewNonGModification						                 AS INT            ) AS P_NewNonGModification						            
			,  CAST(P_ModifiedPaymentAmountOp2							         AS INT            ) AS P_ModifiedPaymentAmountOp2							    
			,  CAST(P_ModifiedPaymentAmountOp3						             AS INT            ) AS P_ModifiedPaymentAmountOp3						        
			,  CAST(NP_Denouncement			                                     AS INT            ) AS NP_Denouncement			                                
			,  CAST(NP_MissedPayment				                             AS INT            ) AS NP_MissedPayment				                        
			,  CAST(NP_InsufficientPayment				                         AS INT            ) AS NP_InsufficientPayment				                    
			,  CAST(GModified				                                     AS INT            ) AS GModified				                                
			,  CAST(ObjectiveStatus							                     AS STRING         ) AS ObjectiveStatus							                
			,  CAST(InstallmentAfterModification					             AS DECIMAL(19,2)  ) AS InstallmentAfterModification					        
			,  CAST(PaymentsAfterModification					                 AS DECIMAL(19,2)  ) AS PaymentsAfterModification					            
			,  CAST(Portfolio    						                         AS STRING         ) AS PortfolioName						                    
			,  CAST(Category				                                     AS STRING         ) AS Category				                                
			,  CAST(PayerStartingPeriod 						                 AS DECIMAL(19,2)  ) AS PayerStartingPeriod 						            
			,  CAST(PayerNoGInflows						                         AS DECIMAL(19,2)  ) AS PayerNoGInflows						                    
			,  CAST(PayerNoGNoAndAmountOfPaymentsInflows						 AS DECIMAL(19,2)  ) AS PayerNoGNoAndAmountOfPaymentsInflows					
			,  CAST(PayerNoGAmountOfPaymentsInflows					             AS DECIMAL(19,2)  ) AS PayerNoGAmountOfPaymentsInflows					        
			,  CAST(PayerNoGBucket0Inflows					                     AS DECIMAL(19,2)  ) AS PayerNoGBucket0Inflows					                
			,  CAST(PayerNoGOtherReasonInflows	                                 AS DECIMAL(19,2)  ) AS PayerNoGOtherReasonInflows	                            
			,  CAST(PayerGInflows			                                     AS DECIMAL(19,2)  ) AS PayerGInflows			                                
			,  CAST(PayerGModificationInflows			                         AS DECIMAL(19,2)  ) AS PayerGModificationInflows			                    
			,  CAST(PayerGAmountOfPaymentsInflows			                     AS DECIMAL(19,2)  ) AS PayerGAmountOfPaymentsInflows			                
			,  CAST(PayerOutflows					                             AS DECIMAL(19,2)  ) AS PayerOutflows					                        
			,  CAST(PayerDenouncementOutflows			                         AS DECIMAL(19,2)  ) AS PayerDenouncementOutflows			                    
			,  CAST(PayerMissedPaymentsOutflows						             AS DECIMAL(19,2)  ) AS PayerMissedPaymentsOutflows						        
			,  CAST(PayerMissedPaymentsPaymentShockOutflows						 AS DECIMAL(19,2)  ) AS PayerMissedPaymentsPaymentShockOutflows					
			,  CAST(PayerOtherReasonOutflows			                         AS DECIMAL(19,2)  ) AS PayerOtherReasonOutflows			                    
			,  CAST(PayerPayments				                                 AS DECIMAL(19,2)  ) AS PayerPayments				                            
			,  CAST(PayerHaircut					                             AS DECIMAL(19,2)  ) AS PayerHaircut					                        
			,  CAST(PayerLiquidations					                         AS DECIMAL(19,2)  ) AS PayerLiquidations					                    
			,  CAST(PayerSales						                             AS DECIMAL(19,2)  ) AS PayerSales						                        
			,  CAST(PayerEndingPeriod					                         AS DECIMAL(19,2)  ) AS PayerEndingPeriod					                    
			,  CAST(NonPayerStartingPeriod						                 AS DECIMAL(19,2)  ) AS NonPayerStartingPeriod						            
			,  CAST(NonPayerInflows                                    			 AS DECIMAL(19,2)  ) AS NonPayerInflows                                         
			,  CAST(NonPayerOutflows                                   			 AS DECIMAL(19,2)  ) AS NonPayerOutflows                                        
			,  CAST(NonPayerPayments                                   			 AS DECIMAL(19,2)  ) AS NonPayerPayments                                        
			,  CAST(NonPayerHaircut                                              AS DECIMAL(19,2)  ) AS NonPayerHaircut                                         
			,  CAST(NonPayerLiquidations                                         AS DECIMAL(19,2)  ) AS NonPayerLiquidations                                    
			,  CAST(NonPayerWriteoffs                                            AS DECIMAL(19,2)  ) AS NonPayerWriteoffs                                       
			,  CAST(NonPayerSales                                                AS DECIMAL(19,2)  ) AS NonPayerSales                                           
			,  CAST(NonPayerEndingPeriod                                         AS DECIMAL(19,2)  ) AS NonPayerEndingPeriod                                    
			,  CAST(PayerPerimeterDecrease                                       AS DECIMAL(19,2)  ) AS PayerPerimeterDecrease                                  
			,  CAST(PayerPerimeterIncrease                                       AS DECIMAL(19,2)  ) AS PayerPerimeterIncrease                                  
			,  CAST(PayerBalanceIncrease                                         AS DECIMAL(19,2)  ) AS PayerBalanceIncrease                                    
			,  CAST(NonPayerPerimeterDecrease                                    AS DECIMAL(19,2)  ) AS NonPayerPerimeterDecrease                               
			,  CAST(NonPayerPerimeterIncrease                                    AS DECIMAL(19,2)  ) AS NonPayerPerimeterIncrease                               
			,  CAST(NonPayerAccruedInterest                                      AS DECIMAL(19,2)  ) AS NonPayerAccruedInterest  
			,  CAST(CommonBalanceDiffrence                                       AS DECIMAL(19,2)  ) AS CommonBalanceDiffrence                              
			,  CAST(getdate()                                       			 AS TIMESTAMP  	   ) AS InsertTimeStamp                              
		FROM final_0
'''
) 
final_df.createOrReplaceTempView("final_df")     

In [0]:
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path) # in order to write in delta_lake

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = []
check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="warning")

In [0]:
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
PayerNonPayerWaterfall = spark.sql(
f'''
select * from {delivery_table_path}
where BusinessDate = {run_date!r}
'''
)
PayerNonPayerWaterfall.createOrReplaceTempView("PayerNonPayerWaterfall")

In [0]:
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/HAPS/PayerNonPayerWaterfall/" # 
file_name = "PayerNonPayerWaterfall_" +run_date+ ".csv"
PayerNonPayerWaterfall.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}") # 
rename_and_cleanup_files(output_path, file_name)